## Partial Dependence Plots (PDP)

## Import requisite libraries

In [ ]:
import pandas as pd
from model_tuner import loadObjects
from model_metrics import plot_3d_pdp

from core.functions import load_model_from_mlflow

## Load model and data

In [ ]:
# flavor (run name in MLflow) -> algo prefix used in the artifact folder
FLAVORS = {
    "lr_orig_training":              "lr",
    "lasso_orig_rfe_training":       "lasso",
    "ridge_orig_rfe_training":       "ridge",
    "elastic_net_orig_rfe_training": "elastic_net",
    "xgb_orig_training":             "xgb",
    "cat_orig_training":             "cat",
}

# ----------------------------------------------------------------------
# load all models
# ----------------------------------------------------------------------
models = {
    flavor: load_model_from_mlflow(flavor, algo)
    for flavor, algo in FLAVORS.items()
}

# keep the original short names working for downstream code
model_linear_reg      = models["lr_orig_training"]
model_lasso_rfe       = models["lasso_orig_rfe_training"]
model_ridge_rfe       = models["ridge_orig_rfe_training"]
model_elastic_net_rfe = models["elastic_net_orig_rfe_training"]
model_xgb             = models["xgb_orig_training"]
model_cat             = models["cat_orig_training"]

In [ ]:
label_map = (None,)
apply_label_map = (True,)


label_replacements = {
    # Sub Event Types
    "Shelling/artillery/missile attack": "Shelling / Missile",
    "Peaceful protest": "Peaceful protest",
    "Mob violence": "Mob violence",
    "Government regains territory": "Gov regains territory",
    "Change to group/activity": "Group change",
    "Armed clash": "Armed clash",
    "Abduction/forced disappearance": "Forced disappearance",
    # Source Scale
    "Subnational-Regional": "Subnational Regional",
    "Subnational": "Subnational",
    "Other-Subnational": "Other Subnational",
    "Other-National": "Other National",
    "National": "National",
    "Other-International": "Other International",
    "International": "International",
    "New media-National": "New media National",
    "New media-International": "New media International",
}

In [ ]:
source_scale_map = {
    "International": "Intl",
    "Local partner-New media": "Local+Media",
    "National": "National",
    "National-International": "Nat+Intl",
    "National-Regional": "Nat+Reg",
    "New media": "Media",
    "New media-National": "Media+Nat",
    "Other": "Other",
    "Other-International": "Other+Intl",
    "Other-National": "Other+Nat",
    "Other-New media": "Other+Media",
    "Other-Regional": "Other+Reg",
    "Other-Subnational": "Other+Subnat",
    "Regional": "Regional",
    "Subnational": "Subnat",
    "Subnational-National": "Subnat+Nat",
    "Subnational-Regional": "Subnat+Reg",
}

event_map = {
    "Abduction/forced disappearance": "Abduction",
    "Agreement": "Agreement",
    "Air/drone strike": "Airstrike",
    "Armed clash": "Armed Clash",
    "Arrests": "Arrests",
    "Attack": "Attack",
    "Change to group/activity": "Group Change",
    "Chemical weapon": "Chemical",
    "Disrupted weapons use": "Disrupted Use",
    "Government regains territory": "Govt Gains",
    "Grenade": "Grenade",
    "Looting/property destruction": "Looting",
    "Mob violence": "Mob Violence",
    "Non-state actor overtakes territory": "NSA Gains",
    "Other": "Other",
    "Peaceful protest": "Peaceful Protest",
    "Remote explosive/landmine/IED": "IED",
    "Sexual violence": "Sexual Violence",
    "Shelling/artillery/missile attack": "Shelling",
    "Suicide bomb": "Suicide Bomb",
    "Violent demonstration": "Violent Protest",
}

In [ ]:
X_train = pd.read_parquet("../data/processed/X_train.parquet")
y_train = pd.read_parquet("../data/processed/y_train_log_fatalities.parquet")
X_valid = pd.read_parquet("../data/processed/X_valid.parquet")
y_valid = pd.read_parquet("../data/processed/y_valid_log_fatalities.parquet")
X_test = pd.read_parquet("../data/processed/X_test.parquet")
y_test = pd.read_parquet("../data/processed/y_test_log_fatalities.parquet")

In [ ]:
# Get bare estimator and transformed data
xgb_estimator = model_xgb.estimator[-1]
preprocessor = model_xgb.estimator[:-1]

feature_names_out = preprocessor.get_feature_names_out()
X_test_transformed = pd.DataFrame(
    (
        preprocessor.transform(X_test).toarray()
        if hasattr(preprocessor.transform(X_test), "toarray")
        else preprocessor.transform(X_test)
    ),
    columns=feature_names_out,
    index=X_test.index,
)

# Pick one representative OHE column per category group
# e.g. use the index of the category in the OHE expansion
sub_event_cols = [f for f in feature_names_out if "sub_event_type" in f]
source_scale_cols = [f for f in feature_names_out if "source_scale" in f]

print("sub_event_type OHE cols:", sub_event_cols)
print("source_scale OHE cols:", source_scale_cols)

In [ ]:
plot_3d_pdp(
    model=xgb_estimator,
    dataframe=X_test_transformed,
    feature_names=[sub_event_cols[0], source_scale_cols[0]],
    x_label="",
    y_label="",
    x_label_map=event_map,
    y_label_map=source_scale_map,
    matplotlib_colormap="viridis",
    z_label="Partial Dependence",
    title="3D Partial Dependence Plot of Event Type vs. Source Scale",
    html_file_path="../data",
    # image_filename="3d_pdp",
    html_file_name="3d_pdp.html",
    image_path_svg="../data",
    plot_type="static",
    text_wrap=80,
    zoom_out_factor=1.3,
    grid_resolution=35,
    label_fontsize=8,
    tick_fontsize=6,
    title_x=0.38,
    figsize=(15, 8),
    top_margin=20,
    # bottom_margin=120,
    right_margin=80,
    view_angle=(23, 50),
    left_margin=70,
    cbar_x=0.9,
    cbar_thickness=25,
    show_modebar=False,
    enable_zoom=True,
    save_plots="html",
)

In [ ]:
model_xgb.get_feature_names()

In [ ]:
from model_metrics import plot_3d_pdp

plot_3d_pdp(
    model=xgb_estimator,
    dataframe=X_test_transformed,
    feature_names=[sub_event_cols[0], source_scale_cols[0]],
    x_label="",
    y_label="",
    # x_label="sub_event_type",
    # y_label="source_scale",
    x_label_map=event_map,
    y_label_map=source_scale_map,
    z_label="Partial Dependence",
    title="3D Partial Dependence Plot of Event Type vs. Source Scale",
    html_file_path="../data",
    # image_filename="3d_pdp",
    html_file_name="3d_pdp.html",
    image_path_svg="../data",
    plot_type="interactive",
    text_wrap=80,
    zoom_out_factor=1.3,
    grid_resolution=25,
    label_fontsize=8,
    tick_fontsize=6,
    title_x=0.38,
    top_margin=20,
    # bottom_margin=120,
    right_margin=80,
    left_margin=70,
    cbar_x=0.9,
    cbar_thickness=25,
    show_modebar=True,
    enable_zoom=True,
    save_plots="html",
    modebar_image_format="svg",
)